# FestGPT — Pipeline RAG 100% Open Source

Este notebook implementa un pipeline RAG (Retrieval-Augmented Generation) completo y reproducible para crear un asistente LLM especializado en festivales musicales.

## Requisitos previos

1. **Crear entorno virtual con Python 3.11:**
```bash
python3.11 -m venv .venv
source .venv/bin/activate
```

2. **Instalar dependencias:**
```bash
pip install -r requirements.txt
```

3. **Registrar el kernel de Jupyter:**
```bash
python -m ipykernel install --user --name festgpt --display-name "FestGPT (3.11)"
```

4. **Configurar token de HuggingFace:**
```bash
export HF_TOKEN="tu_token_aquí"
```

## 0. Configuración del entorno

Selecciona el modo de ejecución antes de ejecutar el resto del notebook:

| Parámetro | `"local"` | `"server"` |
|-----------|-----------|------------|
| Modelo LLM | TinyLlama 1.1B Chat | Phi-3-mini 3.8B (4k) |
| Dispositivo | CPU | CUDA (GPU única) |
| Cuantización | No (modelo ya ligero) | FP16 / BF16 nativo |
| `max_new_tokens` | 256 | 1024 |
| Embeddings | CPU | CUDA |

> **Cambiar `ENV_MODE` en la celda siguiente es lo único que hay que tocar.**

In [3]:
import torch

# ============================================================
#  CONFIGURACIÓN: Cambia esta variable según tu entorno
# ============================================================
# "local"  → CPU, TinyLlama 1.1B (ligero, para pruebas)
# "server" → GPU, Phi-3-mini 3.8B (completo, para producción)
# ============================================================
ENV_MODE = "local"  # <-- CAMBIAR AQUÍ: "local" o "server"
# ============================================================

assert ENV_MODE in ("local", "server"), f"ENV_MODE inválido: {ENV_MODE}"

# --- Configuración derivada ---
if ENV_MODE == "local":
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ~1.1B params, cabe en CPU
    DEVICE = "cpu"
    DEVICE_MAP = "cpu"
    TORCH_DTYPE = torch.float32
    USE_QUANTIZATION = False    # TinyLlama ya es ligero
    MAX_NEW_TOKENS = 256
    BATCH_SIZE = 1
    EMBEDDING_DEVICE = "cpu"
else:  # server (GPU única)
    MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"  # ~3.8B params
    DEVICE = "cuda:0"
    DEVICE_MAP = {"": 0}        # todo el modelo en GPU 0
    TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    USE_QUANTIZATION = False    # precisión nativa en GPU
    MAX_NEW_TOKENS = 1024
    BATCH_SIZE = 4
    EMBEDDING_DEVICE = "cuda:0"

print(f"Modo: {ENV_MODE.upper()}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Dispositivo: {DEVICE}")
print(f"  Dtype: {TORCH_DTYPE}")
print(f"  Max tokens: {MAX_NEW_TOKENS}")
if ENV_MODE == "server" and torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU: {torch.cuda.get_device_name(0)} ({gpu.total_mem / 1e9:.1f} GB)")

Modo: LOCAL
  Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Dispositivo: cpu
  Dtype: torch.float32
  Max tokens: 256


## 1. Modelo de Embeddings

Cargamos el modelo `BAAI/bge-small-en` para generar representaciones vectoriales de los textos.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": EMBEDDING_DEVICE}
)
print(f"Modelo de embeddings cargado en {EMBEDDING_DEVICE} ✓")

/home/aurorax/.virtualenvs/general/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3033.95it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embeddings cargado en cpu ✓


## 2. Carga de documentos y vector store

Cargamos los documentos `.txt` del festival desde la carpeta local `festival_txts/warm_up`, los dividimos en chunks y los almacenamos en una base de datos vectorial ChromaDB persistida en disco.

In [6]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Ruta local a los documentos del festival
DATA_DIR = Path("./festival_txts/warm_up")
CHROMA_DIR = Path("./chroma_db")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontró la carpeta de datos: {DATA_DIR.resolve()}")

loader = DirectoryLoader(str(DATA_DIR), glob="*.txt")
docs = loader.load()
print(f"Documentos cargados: {len(docs)}")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"Chunks generados: {len(chunks)}")

# Vector store persistido en disco
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="festival_rag",
    persist_directory=str(CHROMA_DIR)
)
print(f"Vector store creado y persistido en {CHROMA_DIR.resolve()} ✓")

Documentos cargados: 4
Chunks generados: 4
Vector store creado y persistido en /home/aurorax/Git_repos/ryc/PHDS/Maria/chatbotTesis/chroma_db ✓


## 3. Autenticación en HuggingFace

Necesario para descargar el modelo Phi-3. El token se lee de la variable de entorno `HF_TOKEN` o se solicita de forma segura.

In [7]:
import os
import getpass
from huggingface_hub import login

# Token desde variable de entorno o input seguro
token = os.environ.get("HF_TOKEN") or getpass.getpass("Introduce tu token de HuggingFace: ")
login(token=token)
print("Autenticación en HuggingFace Hub ✓")

Autenticación en HuggingFace Hub ✓


## 4. Carga del modelo LLM

El modelo se selecciona automáticamente según el modo:
- **Local**: `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — 1.1B parámetros, funciona en CPU sin problemas.
- **Server**: `microsoft/Phi-3-mini-4k-instruct` — 3.8B parámetros, precisión nativa en GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print(f"Descargando y cargando: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

load_kwargs = {
    "torch_dtype": TORCH_DTYPE,
    "device_map": DEVICE_MAP,
}

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.3,
    do_sample=True,
    return_full_text=False,
)

print(f"Modelo '{MODEL_NAME.split('/')[-1]}' cargado en modo {ENV_MODE.upper()} 🎉")

Descargando y cargando: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


`torch_dtype` is deprecated! Use `dtype` instead!


## 5. Configuración del Retriever

Configuramos el retriever para buscar los 4 chunks más similares a cada consulta.

In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

## 6. Función de generación RAG

Función principal que combina retrieval + generación. El formato del prompt se adapta automáticamente al modelo:
- **TinyLlama** (local): formato Zephyr (`<|system|>`, `<|user|>`, `<|assistant|>`)
- **Phi-3** (server): formato Phi-3 (`<|user|>`, `<|end|>`, `<|assistant|>`)

In [ ]:
SYSTEM_MSG = """Eres un asistente experto en festivales musicales. Responde SIEMPRE en español.
Usa ÚNICAMENTE la información del contexto proporcionado para responder.
Si la información no es suficiente para dar una respuesta precisa, indícalo claramente.
Sé conciso y útil."""


def _build_prompt(context: str, query: str) -> str:
    """Construye el prompt con el formato de chat adecuado al modelo."""
    if "TinyLlama" in MODEL_NAME:
        # Formato Zephyr (usado por TinyLlama-Chat)
        return (
            f"<|system|>\n{SYSTEM_MSG}</s>\n"
            f"<|user|>\nCONTEXTO:\n{context}\n\nPREGUNTA: {query}</s>\n"
            f"<|assistant|>\n"
        )
    else:
        # Formato Phi-3
        return (
            f"<|user|>\n{SYSTEM_MSG}\n\n"
            f"CONTEXTO:\n{context}\n\n"
            f"PREGUNTA: {query}\n<|end|>\n<|assistant|>"
        )


def generate_answer(query: str) -> str:
    """Genera una respuesta RAG usando el retriever y el LLM."""
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs])
    prompt = _build_prompt(context, query)

    try:
        output = llm_pipe(prompt)[0]["generated_text"]
        return output.strip()
    except Exception as e:
        return f"Error al generar respuesta: {e}"

## 7. Prueba del pipeline

Probamos el asistente con una pregunta de ejemplo.

In [ ]:
print(generate_answer("¿Qué autobuses llegan al festival?"))

## 8. Limpieza de recursos

Libera la memoria GPU/CPU ocupada por el modelo. Ejecutar al terminar la sesión.

In [ ]:
import gc
import torch

# Liberar modelo y tokenizer de memoria
del model
del tokenizer
del llm_pipe

# Limpiar caché de GPU si está disponible
if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()
print("Recursos liberados ✓")